In [38]:
import pandas as pd
import numpy as np
import random
import time
from math import floor, ceil

In [39]:
INVENTORY_CSV = "inventory__modified_dataset.csv"
INVOICES_CSV = "invoices_10k.csv"
OUTPUT_CSV = "invoice_line_items.csv"
N_MIN_LINES = 1
N_MAX_LINES = 5
RANDOM_SEED = 42
PROGRESS_EVERY = 500

np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

# load data
inv = pd.read_csv(INVENTORY_CSV, dtype={'product_id': str})
inv['unit_price'] = inv['unit_price'].astype(float).round(2)
products = inv[['product_id', 'unit_price']].set_index('product_id')['unit_price'].to_dict()
product_ids = list(products.keys())

invoices = pd.read_csv(INVOICES_CSV, dtype={'invoice_id': str})
INVOICE_TOTAL_COL = "subtotal"
if INVOICE_TOTAL_COL not in invoices.columns:
    raise ValueError(f"Invoice total column '{INVOICE_TOTAL_COL}' not found in {INVOICES_CSV}")

In [40]:
# add weights
n_products = len(product_ids)
n_rare = max(1, int(0.10 * n_products))
rare_set = set(np.random.choice(product_ids, size=n_rare, replace=False))
weights = np.array([0.05 if pid in rare_set else 1.0 for pid in product_ids], dtype=float)
n_popular = max(3, int(0.02 * n_products))
popular = list(np.random.choice(product_ids, size=n_popular, replace=False))
for p in popular:
    weights[product_ids.index(p)] *= 4.0
prob = weights / weights.sum()

def distribute_targets(pre_values, target_total):
    pre = np.array(pre_values, dtype=float)
    if pre.sum() == 0:
        raw = np.full(len(pre), target_total / max(1, len(pre)))
    else:
        raw = pre / pre.sum() * target_total
    rounded = np.round(raw, 2)
    diff = round(target_total - rounded.sum(), 2)
    i = 0
    while abs(diff) >= 0.01:
        rounded[i % len(rounded)] += 0.01 if diff > 0 else -0.01
        diff = round(target_total - rounded.sum(), 2)
        i += 1
    return rounded.tolist()

def allocate_quantities_from_targets(unit_prices, targets, max_adjust_per_line=5):
    n = len(unit_prices)
    # initial quantities: round(target / unit_price) but at least 1
    qtys = []
    for up, tgt in zip(unit_prices, targets):
        if up <= 0:
            q = 1
        else:
            q = max(1, int(floor(tgt / up)))
        qtys.append(q)

    # compute totals and residual
    line_totals = [round(q * up, 2) for q, up in zip(qtys, unit_prices)]
    current_sum = round(sum(line_totals), 2)
    target_sum = round(sum(targets), 2)
    residual = round(target_sum - current_sum, 2)

    # if residual is less than 0.01, return immediately
    if abs(residual) < 0.01:
        return qtys, line_totals

    # sorted by unit_price
    idx_desc = sorted(range(n), key=lambda i: unit_prices[i], reverse=True)
    idx_asc = sorted(range(n), key=lambda i: unit_prices[i])

    # if we need to increase sum, add units to highest unit_price lines first
    if residual > 0:
        for i in idx_desc:
            if residual <= 0:
                break
            up = unit_prices[i]
            if up <= 0:
                continue

            max_inc = max_adjust_per_line
            for _ in range(max_inc):
                qtys[i] += 1
                line_totals[i] = round(qtys[i] * up, 2)
                current_sum = round(sum(line_totals), 2)
                residual = round(target_sum - current_sum, 2)
                if residual <= 0:
                    break
    else:
        for i in idx_asc:
            if residual >= 0:
                break
            while qtys[i] > 1 and residual < 0:
                qtys[i] -= 1
                line_totals[i] = round(qtys[i] * unit_prices[i], 2)
                current_sum = round(sum(line_totals), 2)
                residual = round(target_sum - current_sum, 2)
                if residual >= 0:
                    break

    # final recompute and tiny residual fix
    line_totals = [round(q * up, 2) for q, up in zip(qtys, unit_prices)]
    final_sum = round(sum(line_totals), 2)
    diff = round(target_sum - final_sum, 2)
    if abs(diff) >= 0.01:
        
        idx = int(max(range(n), key=lambda i: unit_prices[i] if unit_prices[i] > 0 else -1))
        line_totals[idx] = round(line_totals[idx] + diff, 2)

    # negative check
    for i in range(n):
        if line_totals[i] < 0:
            line_totals[i] = 0.00
            qtys[i] = max(1, qtys[i])

    return qtys, line_totals

In [41]:
rows = []
line_id = 1
invoice_count = 0
start_time = time.time()

for idx, inv_row in enumerate(invoices.itertuples(index=False), start=1):
    invoice_id = str(inv_row.invoice_id)
    invoice_total = float(getattr(inv_row, INVOICE_TOTAL_COL))

    # choose number of lines for this invoice
    n_lines = int(np.random.randint(N_MIN_LINES, N_MAX_LINES + 1))

    # pick products
    chosen_products = list(np.random.choice(product_ids, size=n_lines, p=prob, replace=True))
    unit_prices = [products[pid] for pid in chosen_products]

    # compute proportional targets
    pre_values = [up for up in unit_prices]
    targets = distribute_targets(pre_values, invoice_total)

    # allocate integer quantities from targets
    qtys, line_totals = allocate_quantities_from_targets(unit_prices, targets)

    # ensure invoice sum equals invoice_total exactly
    current_sum = round(sum(line_totals), 2)
    residual = round(invoice_total - current_sum, 2)
    if abs(residual) >= 0.01:

        j = int(np.argmax(line_totals))
        line_totals[j] = round(line_totals[j] + residual, 2)

        up = unit_prices[j]
        if up > 0:
            new_qty = max(1, int(floor(line_totals[j] / up)))
            if new_qty != qtys[j]:
                qtys[j] = new_qty
                line_totals[j] = round(qtys[j] * up, 2)
                current_sum = round(sum(line_totals), 2)
                residual = round(invoice_total - current_sum, 2)
                if abs(residual) >= 0.01:
                    line_totals[j] = round(line_totals[j] + residual, 2)

    # safety clamps
    for i in range(len(qtys)):
        if qtys[i] < 1:
            qtys[i] = 1
        if unit_prices[i] <= 0:
            unit_prices[i] = 0.01
        if line_totals[i] < 0:
            line_totals[i] = 0.00

    # rounding fix
    final_sum = round(sum(line_totals), 2)
    diff = round(invoice_total - final_sum, 2)
    if abs(diff) >= 0.01:
        line_totals[-1] = round(line_totals[-1] + diff, 2)

    # append rows
    for pid, q, up, total in zip(chosen_products, qtys, unit_prices, line_totals):
        rows.append({
            'line_item_id': line_id,
            'invoice_id': invoice_id,
            'product_id': pid,
            'quantity': int(q),
            'unit_price': round(up, 2),
            'discount': 0.00,
            'total_amount': round(total, 2)
        })
        line_id += 1

    invoice_count += 1
    if invoice_count % PROGRESS_EVERY == 0:
        elapsed = time.time() - start_time
        print(f"Processed {invoice_count}/{len(invoices)} invoices — elapsed {elapsed:.1f}s")

Processed 500/10000 invoices — elapsed 0.1s
Processed 1000/10000 invoices — elapsed 0.3s
Processed 1500/10000 invoices — elapsed 0.5s
Processed 2000/10000 invoices — elapsed 0.6s
Processed 2500/10000 invoices — elapsed 0.7s
Processed 3000/10000 invoices — elapsed 0.9s
Processed 3500/10000 invoices — elapsed 1.0s
Processed 4000/10000 invoices — elapsed 1.2s
Processed 4500/10000 invoices — elapsed 1.3s
Processed 5000/10000 invoices — elapsed 1.5s
Processed 5500/10000 invoices — elapsed 1.6s
Processed 6000/10000 invoices — elapsed 1.7s
Processed 6500/10000 invoices — elapsed 1.9s
Processed 7000/10000 invoices — elapsed 2.0s
Processed 7500/10000 invoices — elapsed 2.2s
Processed 8000/10000 invoices — elapsed 2.3s
Processed 8500/10000 invoices — elapsed 2.4s
Processed 9000/10000 invoices — elapsed 2.6s
Processed 9500/10000 invoices — elapsed 2.7s
Processed 10000/10000 invoices — elapsed 2.8s


In [42]:
# save csv
df_out = pd.DataFrame(rows, columns=['line_item_id','invoice_id','product_id','quantity','unit_price','discount','total_amount'])
df_out.to_csv(OUTPUT_CSV, index=False)
elapsed_total = time.time() - start_time
print(f"Wrote {len(df_out)} line items for {invoice_count} invoices to {OUTPUT_CSV} (elapsed {elapsed_total:.1f}s)")

Wrote 30108 line items for 10000 invoices to invoice_line_items.csv (elapsed 4.0s)


In [43]:
# per-invoice totals check
negatives = df_out[df_out['total_amount'] < 0]
print("Negative total_amount rows:", len(negatives))

merged = df_out.groupby('invoice_id', as_index=False)['total_amount'].sum().rename(columns={'total_amount':'lines_total'})
check = invoices.copy()
check['invoice_id'] = check['invoice_id'].astype(str)
check = check.merge(merged, on='invoice_id', how='left')
check['lines_total'] = check['lines_total'].fillna(0.0)
check['diff'] = (check[INVOICE_TOTAL_COL].astype(float) - check['lines_total']).round(2)
mismatches = check[check['diff'].abs() > 0.01]
print("Invoices with mismatched totals (> $0.01):", len(mismatches))
if len(mismatches) > 0:
    print(mismatches.head().to_string(index=False))


Negative total_amount rows: 7
Invoices with mismatched totals (> $0.01): 0


In [44]:
#per-invoice totals match
merged = df_out.groupby('invoice_id', as_index=False)['total_amount'].sum().rename(columns={'total_amount':'lines_total'})
check = invoices.copy()
check['invoice_id'] = check['invoice_id'].astype(str)
check = check.merge(merged, on='invoice_id', how='left')
check['lines_total'] = check['lines_total'].fillna(0.0)
check['diff'] = (check[INVOICE_TOTAL_COL].astype(float) - check['lines_total']).round(2)
mismatches = check[check['diff'].abs() > 0.01]
print("Invoices with mismatched totals (> $0.01):", len(mismatches))
if len(mismatches) > 0:
    print(mismatches.head().to_string(index=False))

Invoices with mismatched totals (> $0.01): 0
